# CS336 Assignment 2 · Part 2 实现指南
> **目标**：不依赖 PyTorch 自动微分与 SDPA，从零实现 FlashAttention-2 的 forward/backward，GPU kernel 用 **CUTLASS CuTe DSL** 编写，通过测试并与 PyTorch 基线对比。
>
> **方法论**：每一步只引入一个新变量（协议 → 算法 → 执行 → 优化）；每一步都有“慢但显然正确”的参照；先预测、再测量、再解释差异。

## 1. 总路线
```
M0  梯度管道验证（朴素 attention + 手写 backward）   ← 只验证协议
 │
M1  PyTorch tiled forward（保存 LSE）               ← 只验证前向算法
 │
M2  PyTorch backward（LSE 重算 P、D 技巧）           ← 只验证反向数学，M5 的金标准
 │
M3  CuTe DSL forward（拆成 3a~3h 八个小步）          ← 只验证执行
 │
M4  causal masking + 跳过全 mask tile                ← 结构优化
 │
M5  CuTe DSL backward（加分项，两个 kernel 方案）
 │
M6  tile 调参 + benchmark + 回答问题
```

## 2. 四条实现路线的本质区别

| | `scaled_causal_self_attn` (eager/compile) | M0 朴素 + 手写 bwd | M1/M2 FA2 tiled (PyTorch) | M3–M5 CuTe DSL |
|---|---|---|---|---|
| backward 谁写 | autograd 自动 | **手写** | **手写 + LSE/D 技巧** | 手写 kernel |
| 保存的中间量 | autograd 自动存 `P`，O(N²) | 手动存 `P`，O(N²) | **手动只存 `Q,K,V,O,L`，O(N)** | 同左 |
| 执行引擎 | PyTorch C++ / Inductor | PyTorch C++ | PyTorch C++（Python 循环） | 我们的 cubin |
| **验证目标** | 模型正确性 / 执行效率 | **autograd.Function 协议** | **FA2 算法数学** | **kernel 实现与性能** |

三个关键认知：
1. **`torch.compile` 不会替你发明算法**——它只做自动融合，`P` 矩阵照样保存，O(N²) 显存省不掉；
2. **`autograd.Function` 是 autograd 穿不透的黑盒**——手写 kernel 同样如此，M0 必须先验证协议；
3. **从 M1 到 M3 变的不是数学，是执行模型**——tile 循环从 Python 解释器移进 GPU，中间量从全局内存移进寄存器/SMEM。
## 3. 各里程碑要点
### M0：协议验证（朴素 attention + 手写 backward）
数学用最朴素的，不可能出错；验证对象是 `autograd.Function` 的接口协议。三个最容易踩的坑，代码里已规避：
- backward 里用到的张量（包括 `V`）必须全部 `save_for_backward`；
- 跨阶段使用的量（如 `scale`）存到 `ctx`，不依赖 forward 的局部变量；
- `dQ = (dS @ K) * scale`——dQ 对应的是 K，不是 V。
```python
import math, torch
class NaiveAttentionFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, K, V, is_causal=False):
        d = Q.shape[-1]
        scale = 1.0 / math.sqrt(d)
        S = (Q @ K.transpose(-2, -1)) * scale
        if is_causal:
            mask = torch.triu(torch.ones(Q.shape[-2], Q.shape[-2],
                              dtype=torch.bool, device=Q.device), diagonal=1)
            S = S.masked_fill(mask, float("-inf"))
        P = torch.softmax(S, dim=-1)
        O = P @ V
        ctx.save_for_backward(Q, K, V, P)
        ctx.scale = scale
        return O
    @staticmethod
    def backward(ctx, dO):
        Q, K, V, P = ctx.saved_tensors
        dV = P.transpose(-2, -1) @ dO
        dP = dO @ V.transpose(-2, -1)
        dS = P * (dP - (P * dP).sum(-1, keepdim=True))   # softmax 的雅可比
        dQ = (dS @ K) * ctx.scale
        dK = (dS.transpose(-2, -1) @ Q) * ctx.scale
        return dQ, dK, dV, None                          # 顺序 = forward 参数顺序
```
causal 时被 mask 的位置 `P = 0`，所以 `dS = 0`，backward 不需要再处理 mask。
**验证**：gradcheck（float64、小形状、causal 两种）为最强检查；再与 SDPA/autograd 梯度对比（fp32，容差 1e-4 量级）。
### M1：tiled forward（online softmax + 保存 LSE）
核心：每行维护 running max `m` 与分母部分和 `l`，逐 K-tile 更新并 rescale（乘 `alpha = exp(m_old - m_new)`），最终 `O /= l`、`L = m + log(l)`。

| 坑 | 要点 |
|---|---|
| rescale 顺序 | `O_i` 和 `l_i` 都乘 `alpha`，用**旧** `m_i` 与**新** `m_new` |
| `-inf - (-inf)` = NaN | `m_i` 初值 `-inf`；causal 从 j=0 开始天然安全，**换遍历顺序/跳 tile 后要重新检查** |
| 累加精度 | `S、O_i、l_i、m` 全部 fp32，即使输入 bf16 |
| 边界 tile | N 不是 Bq/Bk 整数倍时，切片自动变短，mask 用全局下标 |
| `L` 的约定 | **自然对数、含 1/√d 缩放**——必须与测试约定对齐（写实现前先读测试） |

**强检查**：tile 大小 `Bq/Bk` 不应改变结果（扫含非整数倍的形状 `N ∈ {16,17,100,128,1000}`）。

### M2：backward（LSE 重算 P + D 技巧）
两个关键恒等式：
```
P_ij = exp(S_ij − L_i)                 # 用保存的 L 重算 P，不需要保存的 P
D_i  = rowsum(dO ⊙ O)_i = Σ_j P_ij·dP_ij   # 化简，免去一次 O(N²) 归约
dS   = P ⊙ (dP − D[:, None])
```
梯度公式：`dV += Pᵀ dO`；`dP = dO Vᵀ`；`dQ += dS K · scale`；`dK += dSᵀ Q · scale`。

| 坑 | 要点 |
|---|---|
| `D` 用的是 **O**（最终输出） | 不是 `P @ V` 的中间量 |
| scale 只乘在 `dQ`、`dK` 上 | `dV` 不乘 |
| dQ 跨 K-tile 累加 | 初始化为零，跨 tile 累加 |
| 累加精度 | 梯度 fp32 累加，最后转回输入 dtype |

**验证顺序**：先对 M0 的朴素 backward（金标准），再对 autograd；不同 tile 大小结果一致。**把中间量（`D`、tile 的 `P`、`dS`）做成可复用函数——M5 要逐 tile 对照。**
### M3–M5：CuTe DSL kernel
M3 拆成八个小步（copy kernel → 单 tile GEMM → softmax → 第二 GEMM → online 循环 → 输出/边界 → 流水线 → 接入 autograd），每步独立验证。两个关键硬件技巧：
- **accumulator 布局直通**：第一个 MMA 的 accumulator（fp32→bf16 转换后）直接作为第二个 MMA 的 A 操作数，不经 SMEM；对不上时先用 SMEM 中转（慢但对）；
- **exp2 融合**：把 `scale·log₂(e)` 融进缩放因子，用 `exp2` 代替 `exp`；注意最终 `L` 要换回**自然对数**，与 forward 保存的约定严格一致（M5 同样遵守）。
M5 采用**两个 kernel** 方案（dK/dV kernel 按 K-tile 并行循环 Q-tile；dQ kernel 按 Q-tile 并行循环 K-tile），无跨 block 原子操作、结果确定；代价是重算一次 S/P。不要先做 atomicAdd 单 kernel 方案。
### M4：causal
分两步做，各自验证：① 对角 tile 逐元素 mask（`k_idx > q_idx` 置 `-inf`）；② 跳过全 mask tile——K 循环上界改为 `ceil((i+1)·Bq / Bk)`（**注意 `Bq ≠ Bk` 时不是 `i+1`**）。跳 tile 前后**结果必须不变**。验收：seq=16384 下 causal 比非 causal 快接近 2 倍。

## 4. CuTe DSL 接入手册
> API 随版本变化较大，**以下接口名以你安装版本的官方文档和 examples 为准**；动手前先跑通官方最小例子，抄它的接口习惯。
### 4.1 kernel 编译与 launch（三件套：张量转换、stream 对接、编译缓存）
```python
import cutlass
import cutlass.cute as cute
@cute.kernel
def flash_fwd_kernel(gQ: cute.Tensor, gK: cute.Tensor, ...):
    ...  # device 端：tiler / copy atom / MMA atom / online softmax
@cute.jit                       # host 侧入口（以官方版本为准）
def flash_fwd(mQ: cute.Tensor, mK: cute.Tensor, ..., stream=None):
    ...
    flash_fwd_kernel(...).launch(grid=(...), block=(...))
def run_flash_fwd(Q, K, V, is_causal):
    Q, K, V = Q.contiguous(), K.contiguous(), V.contiguous()
    O   = torch.empty_like(Q)
    LSE = torch.empty(Q.shape[:-1], dtype=torch.float32, device=Q.device)
    # torch → CuTe tensor：官方惯用 from_dlpack（保持 stride/layout 信息）
    mQ = cute.from_dlpack(Q)  # 示意，具体调用以文档为准
    ...
    # 编译并缓存（按 shape/dtype/layout 特化）；launch 必须用 torch 当前 stream
    cute.compile(...)  # 或 jit 调用内自动编译，以官方模式为准
    flash_fwd(mQ, mK, mV, mO, mLSE, stream=torch.cuda.current_stream().cuda_stream)
    return O, LSE
```
**硬性注意点**：
- **stream 必须对接 PyTorch 当前 stream**，否则与 autograd/其他 op 异步错乱；
- **编译缓存**：第二次调用必须是微秒~毫秒级（验收标准）；benchmark 时先预热并**单独报告编译时间**；
- kernel 要求的 layout（如 K 的转置布局）用 **stride 表达**，避免真实转置拷贝；
- **MMA 精度显式指定**：bf16 输入 + fp32 accumulator；**fp32 输入不能用 bf16 MMA**，需要 TF32 atom（`m16n8k8`，精度 ~1e-3）、fp32 FFMA 路径，或 host 端转 bf16——**先读测试的 dtype 与容差再决定**。

### 4.2 `autograd.Function` 封装（前向/反向各 launch 一个 kernel）
```python
class FlashAttentionFn(torch.autograd.Function):
    @staticmethod
    def forward(ctx, Q, K, V, is_causal):
        O, LSE = run_flash_fwd(Q, K, V, is_causal)
        ctx.save_for_backward(Q, K, V, O, LSE)     # FA2 只存这五个 O(N) 量
        ctx.is_causal = is_causal
        return O
    @staticmethod
    def backward(ctx, dO):
        Q, K, V, O, LSE = ctx.saved_tensors
        dQ, dK, dV = run_flash_bwd(dO, Q, K, V, O, LSE, ctx.is_causal)
        return dQ, dK, dV, None
```
协议规则（M0 已验证）：backward 返回值顺序 = forward 参数顺序；`save_for_backward` 只存 backward 需要的东西；不在 forward 里原地修改输入。薄封装 `flash_attention(q,k,v,is_causal)` 即可像普通层调用；`torch.library.custom_op` + `register_fake` 为可选工程化加分项。

### 4.3 M5 backward 的 kernel 结构

| kernel | 并行方式 | 内层循环 | 输出 |
|---|---|---|---|
| 预处理 | 逐行 | — | `D = rowsum(dO⊙O)`（可先用 torch 算） |
| dK/dV | 每 block 一个 K/V tile | 循环 Q tile 累加 | `dK, dV`（fp32 寄存器累加，最后转 bf16） |
| dQ | 每 block 一个 Q tile | 循环 K/V tile 累加 | `dQ` |

提醒：backward 里 `P = exp(S − L)`，**`L` 约定必须与 forward 完全一致**（含 `exp2`/`log₂e` 融合的换算）；反向寄存器压力大，必要时减小 tile 或 stages。

## 5. 硬件约束：共享内存预算
Q 常驻一份，K 与 V 各有 stages 级缓冲：
```
smem ≈ Bq·d·2 + stages · 2 · Bk·d · 2   字节（bf16/fp16 每元素 2B）
```
上限用实测值：
```python
torch.cuda.get_device_properties(0).shared_memory_per_block_optin
```
（sm_120 上约 **99 KB/block**，ncu 显示配置大小约 102.4 KB，以实测为准。）

**d=64、bf16，单位 KB（`*` = 超限）：**
| Bq | Bk | stages=1 | 2 | 3 |
|---|---|---|---|---|
| 32 | 32 | 12 | 20 | 28 |
| 64 | 32 | 16 | 24 | 32 |
| 64 | 64 | 24 | 40 | 56 |
| 128 | 64 | 32 | 48 | 64 |
| 128 | 128 | 48 | 80 | **112\*** |

**d=128、bf16**：(64,64)→48/80/**112\***；(128,64)→64/96/**128\***；(128,128)→96/**160\***。**fp32/TF32（4B/元素）、d=64**：(64,32)→32/48/64；(64,64)→48/80/**112\***；(128,64)→64/96/**128\***。
**线程划分约束**：一个 warp 负责 16 行 Q（对应 `m16n8k16` 的 M 维），所以 **`num_warps = Bq/16`，与 `Bq` 联动**——调参表里这两列不能独立取值。
## 6. Benchmark 设计要点
### 6.1 调参协议（M6）
- 固定 seq=4096、d=64、bf16、causal 扫 5 组 tile；
- ⚠️ **混淆因素**：batch=1 单头时 grid 只有 N/Bq 个 block，Bq=128 只有 32 个 block，而本卡 70 个 SM——seq=4096 的结果会被“吃不满 SM”主导。**补一组 seq=16384**（Bq=128 也有 128 个 block），把“tile 效率”与“并行度”分开；
- 吃满 SM 的最小 seq（batch=1）：Bq=32 → 2240；Bq=64 → 4480；Bq=128 → 8960；
- 编译失败也是数据（记录失败原因：smem 超限、寄存器溢出、布局不合法）；计时排除编译/JIT（先预热），多次取中位数，同一张表内不混用 `do_bench`（清 L2）与 event 计时口径。

### 6.2 显存与 OOM 预测（先预测后测量，16 GiB 卡、batch=1 单头）

| N | fp32 单个 N×N | bf16 单个 N×N |
|---|---|---|
| 16384 | 1.0 GiB | 0.5 GiB |
| 32768 | 4.0 GiB | 2.0 GiB |
| 65536 | 16.0 GiB | 8.0 GiB |

- forward 瞬态峰值 ≈ **2 个 N×N**（scores 与 softmax 同时存在）：fp32 N=65536 ≈ 32 GiB → **OOM**；bf16 N=65536 ≈ 16 GiB → **临界 OOM**；
- **backward 峰值更高**（保存的 P + dP、dS 等临时量，估计 3~4 个 N×N，需实测）：bf16 下 N=32768 可能已逼近上限——**backward 的 OOM 点通常比 forward 早一档**；
- FA 显存 O(N·d)：N=65536、d=64 约 32 MiB 量级，**线性增长**，这是对比的核心卖点。验证方法：log-log 坐标拟合斜率，FA ≈ 1，基线 ≈ 2。

### 6.3 吞吐与需回答的问题
- FLOPs：forward ≈ `4·N²·d`（causal 约减半），backward ≈ 2.5× forward；
- 吞吐参照：先实测 cuBLAS bf16 大 GEMM 峰值作为 100%，FA 达其 **30–50% 属正常**；
- 与 H100 差距原因（用自己的数据支撑）：带宽 ~0.9 vs 3.35 TB/s；sm_120 无 tcgen05（wgmma/TMA 以实测为准，起步按 SM80 mma 路径）；smem ~99 vs 227 KB；70 vs 132 SM；官方 FA 对 Hopper 深度优化；
- **“小 seq 为什么更慢”的核心答案：grid 太小吃不满 SM**（N=1024、Bq=128 只有 8 block，70 SM 只用 11%），叠加 launch 固定开销、causal 负载不均、PyTorch 基线在小 seq 时 S 矩阵能放进 L2。用“block 数 ÷ SM 数”和“小 seq 耗时趋于常数”来验证；
- 用 ncu 分析自己的 kernel（名字过滤用 kernel 名）：Speed of Light（计算/访存受限）、Occupancy、Warp State stall（等 cp.async 还是等 MMA）、寄存器溢出与 bank conflict。

## 7. Checklist
- [ ] 阶段 −1：CuTe DSL 在 sm_120 上最小 kernel 跑通、缓存生效（**门槛，不过不往下**；备选：Triton 完成主线）
- [ ] 阶段 0：读测试定契约（shape/dtype/容差/L 约定/接入点/非整数倍 seq）→ `notes/fa2_contract.md`
- [ ] M0：gradcheck + SDPA 对比通过（causal 两种）
- [ ] M1：`test_flash_forward_pass_pytorch`；tile 大小不影响结果；不保存 P
- [ ] M2：`test_flash_backward_pytorch`；与 M0 一致；中间量做成可复用函数
- [ ] M3：`test_flash_forward_pass_cute_dsl[False]`；与 M1 的 O、L 一致；编译缓存生效
- [ ] M4：`[True]` 通过；跳 tile 不改变结果；causal 明显加速
- [ ] M5（加分）：`test_flash_backward_cute_dsl[False/True]`；与 M2 中间量对上
- [ ] M6：三张表填完 + 三个问题用数据回答
